<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 11 · Tree-Based Methods and Ensembles

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
We implement decision trees, Random Forests, and Gradient Boosted Trees on the feature panel and link them to portfolio diagnostics.

### Getting Help While Using Tree Models
- **Appendix C** lists scikit-learn APIs for trees/ensembles.
- **Chapter 7** covers feature engineering referenced here.
- **Chapter 8** reminds you how to evaluate strategy returns.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

## 1. Build Feature/Label Tables

We assemble the tree model inputs by computing rolling momentum and volatility features and aligning them with next‑day returns as labels.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=["Date"]).set_index("Date").sort_index().ffill()
log_rets = np.log(prices / prices.shift(1)).dropna()
feature_components = {
    "momentum": prices.pct_change(20),
    "volatility": log_rets.rolling(20).std(),
}
features = pd.concat(feature_components, axis=1).dropna()
labels = log_rets.shift(-1)["AAPL"].rename("label")
common_index = features.index.intersection(labels.dropna().index)
features = features.loc[common_index]
labels = labels.loc[common_index]
features.head()

## 2. Decision Tree Baseline

This baseline single tree provides an interpretable set of if–then rules that approximate the mapping from features to returns.

In [ ]:
split = int(len(features) * 0.7)
X_train, X_test = features.iloc[:split], features.iloc[split:]
y_train, y_test = labels.iloc[:split], labels.iloc[split:]
tree = DecisionTreeRegressor(max_depth=4, random_state=0)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)
mean_squared_error(y_test, tree_pred)

## 3. Random Forest and Gradient Boosting

We then fit Random Forest and Gradient Boosted Trees to reduce variance and capture more subtle nonlinearities than a lone tree can.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=50,
    random_state=0,
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
gbt = GradientBoostingRegressor(random_state=0)
gbt.fit(X_train, y_train)
gbt_pred = gbt.predict(X_test)
pd.Series(
    {
        "tree_mse": mean_squared_error(y_test, tree_pred),
        "rf_mse": mean_squared_error(y_test, rf_pred),
        "gbt_mse": mean_squared_error(y_test, gbt_pred),
    }
)

### 3.1 Feature Importance Comparison

Using built‑in importance scores, we compare how RF and GBT rank the same set of features, which is a first step toward explainability.

In [ ]:
importances = pd.DataFrame(
    {
        "RF": rf.feature_importances_,
        "GBT": gbt.feature_importances_,
    },
    index=features.columns,
)
importances

### Performance Helper
We reuse the Chapter 8 metric function to translate predictions into stats.

In [ ]:
def performance_stats(returns: pd.Series, risk_free: float = 0.02):
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (ann_ret - risk_free) / ann_vol if ann_vol > 0 else np.nan
    wealth = (1 + returns).cumprod()
    max_dd = (wealth / wealth.cummax() - 1).min()
    return pd.Series(
        {
            "annualized_return": ann_ret,
            "annualized_vol": ann_vol,
            "sharpe": sharpe,
            "max_drawdown": max_dd,
        }
    )

## 4. Portfolio Signal from Random Forest Predictions

Finally, we convert Random Forest predictions into cross‑sectional ranks and compute portfolio returns to see whether the model adds value.

In [ ]:
signal = pd.Series(rf_pred, index=X_test.index).rank(pct=True) - 0.5
strategy_returns = signal * y_test
performance_stats(strategy_returns)

## 5. Exercises
### Exercise 1 – Hyperparameter Sweep
Evaluate RF/GBT performance over grids of depth and learning rate.
<details><summary>Hint</summary>
Use <code>RandomizedSearchCV</code> with custom scoring (e.g., negative MSE).
</details>

### Exercise 2 – Permutation Importance
Implement permutation importance for the RF model and compare to built-in feature importances.
<details><summary>Hint</summary>
Use <code>sklearn.inspection.permutation_importance</code> with <code>n_repeats=20</code>.
</details>

### Exercise 3 – Turnover Control
Smooth the RF signal using an exponential moving average before computing strategy returns.
<details><summary>Hint</summary>
Apply <code>.ewm(alpha=0.2).mean()</code> to the signal series.
</details>


## 6. Takeaways for Chapter 11
- Tree ensembles offer nonlinear pattern capture with minimal feature engineering.
- Comparing tree/RF/GBT metrics ensures we balance accuracy with interpretability.
- Translating ensemble scores into portfolio signals exposes turnover and stability considerations.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">